# 2.1 · Data manipulation and tensors

**Machine Learning Foundations for AI Safety · Chapter 2**  
**Estimated time:** 30–40 minutes · **Prerequisite:** knowing what a Python list is.

### What you will be able to do

- Explain a tensor using everyday examples, identify its **shape** and **axes**, and tell a scalar, vector and matrix apart.
- Select a cell or a row/column using Python indexing.
- Explain the difference between *elementwise arithmetic* and *broadcasting*.
- Run small PyTorch examples and predict their outputs before running them.

> **Our running example:** A made-up class of four learners. Each learner has study hours, completed practice questions, and a quiz score. We use synthetic numbers for easy arithmetic; they are not evidence that hours alone determine a real student's score.

**Learning method:** Read the idea → inspect the picture → predict the output → run the code → explain it in your own words.


## 1. What does “data” look like to a computer?

We may write **“Ben studied 2 hours, completed 4 questions, and scored 5.”** A computer can process this observation as three numbers: `[2, 4, 5]`. Put four learners together and we get a rectangular **table**.

A **row** describes *one learner*. A **column** describes *one type of measurement*. A **cell** is where a particular row meets a particular column; every cell stores exactly one value.

Here is the entire table. Count the four learner rows and three numeric columns. The table therefore has shape **(4, 3)**. The names help humans read the data, but the names are not included in this particular numeric matrix.


<picture>
  <source type="image/svg+xml" srcset="images/01-table-rows-columns.svg">
  <img src="images/01-table-rows-columns.png" alt="A fully labeled 4-row × 3-column data table with the cell for Ben’s quiz score highlighted" loading="lazy">
</picture>

*Figure: A fully labeled 4-row × 3-column data table with the cell for Ben’s quiz score highlighted.*


For example, **row Ben + column Score = 5**. The table is not just a picture of cells: it is an organized way to tell the computer which value belongs where.

### Try this in Python

First build the numeric table with PyTorch, a library used widely in deep learning. We will not need a GPU or any downloaded dataset.


In [ ]:
import torch

learner_names = ["Alice", "Ben", "Chika", "Dara"]
data = torch.tensor([
    [1., 2., 3.],  # Alice: hours, practice, score
    [2., 4., 5.],  # Ben
    [3., 6., 7.],  # Chika
    [4., 8., 9.],  # Dara
])

print('Shape:', tuple(data.shape))
print('Number of values:', data.numel())
print('First row:', data[0].tolist())
assert tuple(data.shape) == (4, 3)


**Read the result:** A shape `(4, 3)` means 4 rows and 3 columns; `numel()` returns **12** because 4 × 3 = 12. The individual data values have not changed—only the way we grouped them determines the shape.

## 2. What is a tensor?

A **tensor** is an array of values organized along one or more **axes** (directions). A scalar is a single number; a vector is a one-dimensional array; a matrix is a two-dimensional array. In PyTorch, all three are tensors.

- **Scalar:** one quiz score, `5`; shape `()` (zero axes).
- **Vector:** four scores, `[3, 5, 7, 9]`; shape `(4,)` (one axis).
- **Matrix:** all the numerical observations; shape `(4, 3)` (two axes).
- **Higher-dimensional tensor:** a group of color images can be organized as `(number_of_images, color_channels, height, width)`.

Think of an **axis** as a direction you can move through the data: down the rows (axis 0) or across the columns (axis 1).


### Why do we use tensors instead of separate numbers?

A model usually receives *many examples and many measurements at once*. A tensor stores those values in one organized object so an operation can be applied to an entire group. **Shape describes the arrangement, not the meaning of the values.**

For our dataset, `(4, 3)` means four learners × three numerical measurements. `shape[0] = 4` counts learners, while `shape[1] = 3` counts measurements per learner. A scalar has zero axes, a vector one, and a matrix two. Larger tensors follow the same idea: a batch of images might have shape `(batch, channels, height, width)`.

**Machine-learning connection:** if your model expects three features per example, providing a tensor with four columns will cause a shape mismatch—even if every number looks reasonable.


<picture>
  <source type="image/svg+xml" srcset="images/02-tensor-shapes.svg">
  <img src="images/02-tensor-shapes.png" alt="A scalar, a four-number vector, and a 2 × 3 matrix with every cell shown" loading="lazy">
</picture>

*Figure: A scalar, a four-number vector, and a 2 × 3 matrix with every cell shown.*


In [ ]:
scalar = torch.tensor(5.)
vector = torch.tensor([3., 5., 7., 9.])
matrix = torch.tensor([[1., 2., 3.], [4., 5., 6.]])

for label, tensor in [('scalar', scalar), ('vector', vector), ('matrix', matrix)]:
    print(f'{label:6}: shape={tuple(tensor.shape)}, dimensions={tensor.ndim}')

# Reshaping changes the arrangement, not the individual values.
sequence = torch.arange(12)
print('Reshaped to 3 rows × 4 columns:\n', sequence.reshape(3, 4))


**Predict before running:** `torch.arange(12)` contains 12 numbers, starting at 0 and ending at 11. If we reshape it to `(3, 4)`, we get 3 rows of 4 numbers—not 12 new numbers. A reshape only works when the total number of entries is preserved.

## 3. Finding a particular value: indexing and slicing

Python starts counting positions at **0**, not 1. `data[1, 2]` means **second row, third column**. That is Ben’s score, `5`.

A colon (`:`) says “take all entries along this axis.” So `data[:, 0]` returns **the first column for every learner**. A slice like `data[0:2]` returns rows at positions 0 and 1, stopping *before* 2.


<picture>
  <source type="image/svg+xml" srcset="images/03-indexing.svg">
  <img src="images/03-indexing.png" alt="A rectangular tensor with explicit zero-based row and column indices and one highlighted entry" loading="lazy">
</picture>

*Figure: A rectangular tensor with explicit zero-based row and column indices and one highlighted entry.*


In [ ]:
print('One cell:', data[1, 2].item())
print('All study hours:', data[:, 0].tolist())
print('Ben’s entire row:', data[1].tolist())
print('First two learners:\n', data[0:2])

assert data[1, 2].item() == 5
assert data[:, 0].tolist() == [1., 2., 3., 4.]


## 4. Operations: elementwise vs. broadcasting

### First: what does *elementwise* mean?

With equally shaped tensors, `+` combines **corresponding positions**: `[1, 2] + [10, 20]` becomes `[11, 22]`. Similarly, `*` multiplies positions: `[1, 2] * [10, 20]` becomes `[10, 40]`. No rows or columns are matched by meaning or by name: PyTorch matches their **positions**.

### Broadcasting — precise definition

**Broadcasting is a set of rules that lets PyTorch perform elementwise operations on tensors of different but compatible shapes by treating dimensions of size 1 as expanded to the required size.**

**Why does it exist?** Suppose you have a column containing three students' values and a row containing two adjustments. You want every student to receive both adjustments. You could build two larger tables by hand, but that is tedious and may waste memory. Broadcasting lets you express the same calculation directly.

**Important:** 'expand' means *behave as if values repeat*, not necessarily that PyTorch physically copied the tensor's data. Now follow the original calculation below, and then the expanded view after it.


<picture>
  <source type="image/svg+xml" srcset="images/04-broadcasting.svg">
  <img src="images/04-broadcasting.png" alt="A complete 3 × 1 plus 1 × 2 broadcasting calculation with a full 3 × 2 result table" loading="lazy">
</picture>

*Figure: A complete 3 × 1 plus 1 × 2 broadcasting calculation with a full 3 × 2 result table.*


### How the `(3, 1) + (1, 2)` addition works

**Step 1 — read the shapes.** `A = [[1], [2], [3]]` has **3 rows and 1 column**. `B = [[10, 20]]` has **1 row and 2 columns**.

**Step 2 — imagine expanding only the size-1 dimensions.** A's single column is logically repeated into two columns. B's single row is logically repeated into three rows. Their conceptual views are:

| Row | A column 0 | A column 1 | B column 0 | B column 1 |
|:--|--:|--:|--:|--:|
| 0 | 1 | 1 | 10 | 20 |
| 1 | 2 | 2 | 10 | 20 |
| 2 | 3 | 3 | 10 | 20 |

**Step 3 — add corresponding positions:** top-right `1+20=21`, middle-left `2+10=12`, bottom-right `3+20=23`. The complete output is `[[11,21],[12,22],[13,23]]`, shape `(3,2)`.

Here is the same expansion as fully labeled 3-row × 2-column grids:


<picture>
  <source type="image/svg+xml" srcset="images/04b-broadcasting-expanded.svg">
  <img src="images/04b-broadcasting-expanded.png" alt="Three complete three-row two-column grids: A expands horizontally, B expands vertically, and their elementwise sum is shown cell by cell." loading="lazy">
</picture>

*Figure: Conceptual broadcasting expansion. Colored cells show reused values; no physical copies are required.*


In [ ]:
A = torch.tensor([[1.], [2.], [3.]])   # three rows, one column
B = torch.tensor([[10., 20.]])           # one row, two columns

result = A + B
print('A shape:', tuple(A.shape))
print('B shape:', tuple(B.shape))
print('Result shape:', tuple(result.shape))
print(result)

assert result.tolist() == [[11., 21.], [12., 22.], [13., 23.]]


### Try a compatible shape, then see a shape error

In the cell below we use tiny stand-in values rather than 1,000 learners. Predict the output shape before running. A failed example is caught and reported so it does not stop the notebook.


In [ ]:
subject_scores = torch.tensor([[60., 70., 80.], [75., 85., 95.]])  # 2 learners × 3 subjects
subject_means = torch.tensor([67.5, 77.5, 87.5])           # 3 subject averages
centered = subject_scores - subject_means
print('Scores shape:', tuple(subject_scores.shape))
print('Means shape:', tuple(subject_means.shape))
print('Centered result:\n', centered)
assert centered.shape == (2, 3)

try:
    _ = torch.zeros(3, 2) + torch.zeros(3, 3)
except RuntimeError:
    print('Expected: shapes (3,2) and (3,3) cannot broadcast.')


### When are shapes compatible? Check dimensions from right to left

PyTorch lines up the **last axes first**. Each aligned pair must have the **same size**, or **one size must be 1**, or a leading axis must be absent (treated as 1 for comparison). The result takes the larger size along each axis. If a pair has two different sizes, neither of which is 1, broadcasting fails.

| A shape | B shape | Does it work? | Result and reasoning |
|:--|:--|:--|:--|
| `(3, 1)` | `(1, 2)` | Yes | `(3, 2)` — each axis has a 1 |
| `(3, 2)` | `(3, 2)` | Yes | `(3, 2)` — both axes match |
| `(3, 2)` | `(2,)` | Yes | `(3, 2)` — the vector acts like `(1, 2)` |
| `(3, 2)` | `(3, 3)` | **No** | Last axis: 2 versus 3; neither is 1 |
| `(3, 1, 4)` | `(1, 5, 1)` | Yes | `(3, 5, 4)` — check axes from the right |

**Do not confuse operations:** `A * B` is elementwise multiplication, optionally with broadcasting. `A @ B` is *matrix multiplication*, which uses row–column dot products and its own shape rules (Lesson 2.3).

### Where broadcasting appears in machine learning

Suppose `scores` contains 1,000 learners × 3 subject scores, so it has shape `(1000, 3)`. Suppose `subject_means` contains the three averages, shape `(3,)`. `scores - subject_means` subtracts the relevant subject average from **each learner's row**, without manually creating 1,000 copies of the averages. This pattern is common in feature centering and neural-network bias addition.

**Check the intuition:** Broadcasting changes *how the computation pairs values*. It does **not** invent new observations or determine what the data means.


In [ ]:
experiment = data.clone()  # we create a copy, leaving the original unchanged
experiment[1, 2] = 6.
print('Old Ben score:', data[1, 2].item())
print('Edited Ben score:', experiment[1, 2].item())
print('The edited copy differs from the original:', not torch.equal(data, experiment))  # True


### Explain broadcasting in your own words

Try this four-part explanation aloud: **What** does broadcasting allow? **Why** is it useful? **How** are dimensions compared? **Where** does it appear in machine learning? If you can explain all four using `(3,1) + (1,2)`, you understand the rule rather than memorizing the diagram.


### Check your understanding (answer before revealing by running)

1. A tensor has shape `(5, 4)`. How many **rows**, **columns**, and **values** are there?
2. Which expression selects Chika’s **hours** from `data`?
3. What is the result shape of `(2, 1)` + `(1, 3)`? What are the result’s dimensions?
4. Explain **in one sentence** why `*` is not the same as `@`.

**Practice cell:** Try `data[2, 0]`, `data[:, 1]`, and a broadcast of `[[5], [6]] + [[0, 10, 20]]`.

**Takeaway:** The value of a number and the *position* it occupies in a tensor are different ideas. Understanding shape and indexing prevents many later ML errors.

➡️ **Next:** [2.2 Data preprocessing](02-data-preprocessing.ipynb) — real tables often include missing values and categories.


In [ ]:
# Your turn — edit these expressions and predict their outputs.
print('Chika’s hours:', data[2, 0].item())
print('All practice values:', data[:, 1].tolist())
print('Broadcast example:\n', torch.tensor([[5], [6]]) + torch.tensor([[0, 10, 20]]))


---



# 2.2 · Data preprocessing: from messy to usable

**Estimated time:** 30–40 minutes · **Prerequisite:** Lesson 2.1 (rows, columns and tensors).

### What you will learn

- Explain why a dataset may not be ready for machine learning, even when it looks like a table.
- Identify **missing values** and choose a simple method for handling them.
- Separate **features** (inputs) from the **target** (the answer we want to predict).
- Turn category labels into numbers using **one-hot encoding**, then convert the result into a PyTorch tensor.

We continue with our fictional study dataset. This time, imagine Ben forgot to record the number of practice questions. We have also recorded *how* each learner studied (Video or Text), which is a word rather than a number.


## 1. Why cleaning comes before modeling

A computer needs a consistent representation: values must be in the right columns, missing information must have an explicit treatment, and categorical labels must be encoded appropriately. The picture below shows the transformation we will complete in this notebook.


### What are we trying to achieve?

**Data preprocessing** means turning raw observations into a trustworthy, consistent form the model can use. We do it because raw data may contain unknown values, inconsistent units, text labels, or columns that should *not* be used to predict the answer.

**How it works:** (1) inspect the table, (2) decide how missing values will be handled, (3) separate inputs from the answer, (4) encode categories, and (5) optionally scale numeric features. **ML connection:** a model can learn misleading patterns if preprocessing accidentally uses information from the future or test set.


<picture>
  <source type="image/svg+xml" srcset="images/05-cleaning-pipeline.svg">
  <img src="images/05-cleaning-pipeline.png" alt="Three stages: a raw table, cleaned numeric features, and a numerical tensor" loading="lazy">
</picture>

*Figure: Three stages: a raw table, cleaned numeric features, and a numerical tensor.*


Our starting records are:

| Learner | Hours | Practice | Method | Score / 10 |
|:--|--:|--:|:--|--:|
| Alice | 1 | 2 | Video | 3 |
| Ben | 2 | **missing** | Text | 5 |
| Chika | 3 | 6 | Video | 7 |
| Dara | 4 | 8 | Text | 9 |

A missing value is **not** the same as zero: zero would mean Ben completed *no* practice questions; “missing” means we **do not know**. We must not silently change one into the other.


In [ ]:
import pandas as pd
import torch
from IPython.display import display

raw = pd.DataFrame({
    'Learner': ['Alice', 'Ben', 'Chika', 'Dara'],
    'Hours': [1., 2., 3., 4.],
    'Practice': [2., None, 6., 8.],
    'Method': ['Video', 'Text', 'Video', 'Text'],
    'Score': [3., 5., 7., 9.],
})
display(raw)
print('Missing values per column:\n', raw.isna().sum())


**What did we observe?** `pandas` displays the missing practice value as `NaN` (“Not a Number”). `raw.isna().sum()` counts missing cells in each column. Here, exactly one Practice value is missing.

## 2. Handling a missing value

There is no universally correct replacement. Common choices are:

- **Drop a row** if it cannot be used, though this loses data and can introduce bias.
- **Fill with a meaningful known value** if you really have one.
- **Fill using an observed statistic** such as a median or mean (a simple baseline, not a magic fix).

For demonstration, we fill Ben's practice count using the **mean of the three observed values**:

\[(2+6+8)/3 = 16/3 \approx 5.33.\]

The result is an *estimate*. It is **not** a claim that Ben actually completed 5.33 questions. In a real project, we should record which values were originally missing and evaluate the choice.


### Why choose the mean here, and what could go wrong?

For known counts `2, 6, 8`, the mean is `(2 + 6 + 8) / 3 = 5.33`. Replacing a missing cell with this value is called **mean imputation**. It lets a numerical algorithm accept a complete column, but it introduces an estimated value and may hide differences among learners.

- **Unknown ≠ zero:** zero asserts no practice occurred; missing means the count was unreported.
- **Imputation ≠ recovery:** we have *not* discovered Ben's actual count.
- **Mean ≠ always best:** a median may be more robust to extreme values, and sometimes keeping a separate "was missing" flag is useful.

In real evaluation, compute the fill value on the training portion only. If a test example helps determine the fill value, information from evaluation has leaked into training.


<picture>
  <source type="image/svg+xml" srcset="images/06-missing-values.svg">
  <img src="images/06-missing-values.png" alt="The same four-row table before and after the missing practice value is filled with the observed mean" loading="lazy">
</picture>

*Figure: The same four-row table before and after the missing practice value is filled with the observed mean.*


In [ ]:
cleaned = raw.copy()
observed_mean = cleaned['Practice'].mean()  # NaN is ignored by pandas.mean()
cleaned['Practice'] = cleaned['Practice'].fillna(observed_mean)
print('Observed mean:', round(observed_mean, 2))
display(cleaned)
assert cleaned['Practice'].isna().sum() == 0


**Important for later:** When splitting real data into training and test sets, compute your fill value using **training data only**, then apply that same value to validation/test data. Otherwise information from the test set leaks into model preparation. We'll formalize this in Chapter 3.

## 3. Decide what the model sees—and what it should predict

**Features** are pieces of information given to the model, such as Hours, Practice and Method. The **target** is the desired answer: Score.

We must not accidentally include Score among the model's features while asking it to predict Score—that would give away the answer. We also exclude the learner's name: it is an identifier, not a numeric explanation of performance.


<picture>
  <source type="image/svg+xml" srcset="images/07-feature-target.svg">
  <img src="images/07-feature-target.png" alt="Inputs enter the model; predictions are compared with separately observed target scores, with each table fully labeled" loading="lazy">
</picture>

*Figure: Inputs enter the model; predictions are compared with separately observed target scores, with each table fully labeled.*


**In the illustration**, the predicted scores `[4, 5, 6, 8]` are hypothetical model outputs. They are not known targets. The known scores `[3, 5, 7, 9]` are the targets we compare those predictions against to measure error.


In [ ]:
features = cleaned[['Hours', 'Practice', 'Method']].copy()
target = cleaned['Score'].copy()
print('Features:')
display(features)
print('Target values:', target.to_list())


### Features versus targets: what can the model see?

A **feature** is a measurement available to the model as an input. A **target** is the answer we ask it to predict. In our toy task, study hours, practice count, and study method are input features; Score is the target.

Why keep the target out of the inputs? If we hand a model the correct score while asking it to predict that same score, its excellent apparent performance would be meaningless. This is an obvious case of **target leakage**. Other leakage can be subtler, such as using information collected *after* the outcome.


## 4. Turning categories into numbers

If Method can be `Video` or `Text`, why not replace those with 1 and 2? Because such numbers may wrongly suggest that “Text is twice Video” or that the categories have an order.

**One-hot encoding** makes a separate column for each category. `Video` becomes `[1, 0]`; `Text` becomes `[0, 1]`. The resulting values only say whether a category applies.

For a beginner exercise we keep **both** indicator columns so the transformation is visible. Some models drop one redundant indicator column; that detail can wait until later.


### Read each encoded row, not just the column names

With category columns `Method_Text` and `Method_Video`, a learner marked "Text" has `(1, 0)` and a learner marked "Video" has `(0, 1)` (the order depends on the actual column names). A **1 means the category applies** and **0 means it does not**. We do not interpret the two columns as quantities that should be added or ordered.

**Why this matters:** many models work with numbers but should not be told that Video is "greater than" Text just because we assigned arbitrary numeric labels. Other encoding schemes exist; one-hot is a clear starting point.


<picture>
  <source type="image/svg+xml" srcset="images/08-one-hot-encoding.svg">
  <img src="images/08-one-hot-encoding.png" alt="A fully labeled four-row table showing categorical labels converted to two binary indicator columns" loading="lazy">
</picture>

*Figure: A fully labeled four-row table showing categorical labels converted to two binary indicator columns.*


In [ ]:
numeric_features = pd.get_dummies(features, columns=['Method'], dtype=int)
display(numeric_features)
print('Column names:', numeric_features.columns.to_list())
assert numeric_features.select_dtypes(include='object').empty


## 5. Ready for PyTorch

We can now convert the numeric feature table to a **tensor**. The model would see four rows (learners) and four columns (Hours, Practice, Method_Text, Method_Video): shape **(4, 4)**. The target contains one score per learner: shape **(4,)**.

The `float32` type is widely used for deep-learning calculations. The code below keeps the **inputs and targets separate**.


In [ ]:
X = torch.tensor(numeric_features.to_numpy(dtype='float32'))
y = torch.tensor(target.to_numpy(dtype='float32'))

print('Feature columns:', list(numeric_features.columns))
print('X shape:', tuple(X.shape))
print('y shape:', tuple(y.shape))
print('First learner’s features:', X[0].tolist())
print('First learner’s score:', y[0].item())
assert tuple(X.shape) == (4, 4) and tuple(y.shape) == (4,)


## 6. A first look at scaling

Some numerical features may have very different magnitudes: study hours might range from 1 to 4, while practice questions might range from 0 to 1,000. In many models, large numerical scales can make optimization harder.

One simple technique is **min–max scaling**: subtract the minimum, then divide by the range. For `[1, 2, 3, 4]`, this produces `[0, 0.333…, 0.667…, 1]`. Scaling changes how a number is represented, not which learner it belongs to.

In real training pipelines, just as with filling missing data, learn scaling parameters from **training data only**.


In [ ]:
study_hours = torch.tensor([1., 2., 3., 4.])
scaled_hours = (study_hours - study_hours.min()) / (study_hours.max() - study_hours.min())
print('Before:', study_hours.tolist())
print('After :', [round(v, 3) for v in scaled_hours.tolist()])


### Scaling worked example

**What:** Scaling changes the numerical range of a feature while preserving which rows the values belong to. **Why:** wildly different magnitudes can complicate some optimization procedures. **How:** for min–max scaling, use `(value − minimum) / (maximum − minimum)` when the range is nonzero. For Hours `[1,2,3,4]`, the value 3 becomes `(3−1)/(4−1) = 2/3 ≈ 0.667`.

**A key boundary:** if maximum equals minimum, the denominator is zero, so we must handle that case separately. Also, fitted minimum and maximum must come from **training data**, not from held-out evaluation examples. Scaling is a representation choice, not a guarantee of better predictions.


## 7. Practice: explain every column

Before running the next cell, answer:

1. Which part of our table is the **target**, and why should it not be part of `X`?
2. If `Practice` is missing, is it correct to replace it with `0` automatically? Explain.
3. How many feature columns remain if Method has two categories and we keep both indicator columns?
4. Why is filling missing values using statistics from the **test data** a problem?

**Takeaway:** Preprocessing is the step where raw observations become consistent, model-readable **features and targets**. Decisions here affect everything the model learns.

➡️ **Next:** [2.3 Linear algebra](03-linear-algebra.ipynb) — how models combine those numeric features.


In [ ]:
# Try changing the category encoding: What if a new method "Audio" appeared?
toy = pd.DataFrame({'Method': ['Video', 'Text', 'Audio', 'Video']})
display(pd.get_dummies(toy, dtype=int))
print('Three categories produce three columns with this full one-hot encoding.')


---



# 2.3 · Linear algebra: how a model combines numbers

**Estimated time:** 40–50 minutes · **Prerequisite:** Lesson 2.1 (tensors and shape).

### By the end, you should be able to

- Distinguish a **scalar**, **vector**, **matrix**, and their shapes.
- Calculate a **dot product** by hand and interpret it as a weighted sum.
- Explain a **matrix–vector product**: one prediction per example.
- Explain the shape rule for **matrix multiplication**, and distinguish it from elementwise multiplication.
- Understand **sum**, **mean**, and the **L2 norm** without memorizing formal proofs.

**Running example:** We will use Hours from the fictional learner table to calculate made-up quiz predictions. We deliberately choose a simple rule so each result can be checked by hand.


## 1. Why linear algebra matters to machine learning

Imagine predicting a quiz score from study hours. A simple model could follow the rule:

\[\widehat{\text{Score}} = 2\times \text{Hours} + 1.\]

The **2** is the weight for Hours; the **1** is a bias (constant offset). A larger neural network uses the same essential idea—combine many inputs with learned weights—repeated across many layers.

A convenient way to represent this small example is with two **features** per learner: the number of hours and a constant `1` for the bias. The weights are `[2, 1]`.


## 2. The dot product: one weighted sum

For two vectors of equal length, the **dot product** multiplies matching entries and **adds the products**. Given `x = [2, 4]` and `w = [3, 1]`:

\[x\cdot w = 2\times3 + 4\times1 = 10.\]

Think: each feature contributes its value multiplied by its weight. The final sum is one number (a scalar).


### Why multiplication *and* addition?

A model may consider several pieces of information, each with a different importance or **weight**. The dot product gives one combined value: multiply each feature by its corresponding weight, then add those contributions.

For `[2,4] · [3,1]`: feature 1 contributes `2×3=6`, feature 2 contributes `4×1=4`, and the sum is **10**. The answer is a single scalar, not `[6,4]`.

**ML connection:** a single neuron begins with a weighted sum like this (often plus a bias), before applying its activation function. The weights are learned, not measures of a feature's inherent worth.


<picture>
  <source type="image/svg+xml" srcset="images/09-dot-product.svg">
  <img src="images/09-dot-product.png" alt="A dot product with two explicitly labeled vectors and the full 2×3 + 4×1 calculation" loading="lazy">
</picture>

*Figure: A dot product with two explicitly labeled vectors and the full 2×3 + 4×1 calculation.*


In [ ]:
import torch

x = torch.tensor([2., 4.])
w = torch.tensor([3., 1.])

manual = x[0]*w[0] + x[1]*w[1]
print('Hand-style calculation:', manual.item())
print('PyTorch dot:', torch.dot(x,w).item())
assert torch.dot(x,w).item() == 10.


**Do not confuse `dot` with elementwise multiplication.** `x*w` returns `[6, 4]`, still a vector. `torch.dot(x,w)` **adds** those two values to return the scalar `10`.

A dot product is a building block of predictions, attention scores in Transformers, and many other neural-network operations.


In [ ]:
print('Elementwise x*w:', (x*w).tolist())
print('Dot product x·w:', torch.dot(x,w).item())


## 3. Matrix–vector multiplication: several predictions at once

Instead of predicting one learner at a time, stack the feature vectors as rows of a matrix. Our four learners have Hours `[1,2,3,4]`. For each learner, we append a constant `1`, because the rule includes a bias.

\[
X = \begin{bmatrix}1&1\\2&1\\3&1\\4&1\end{bmatrix},\quad
w=\begin{bmatrix}2\\1\end{bmatrix},\quad
Xw=\begin{bmatrix}3\\5\\7\\9\end{bmatrix}.
\]

Every **output value** is a **row of X dot w**. The first learner gets `1×2 + 1×1 = 3`, the second `2×2 + 1×1 = 5`, and so on.

**Shape rule:** `(4, 2) @ (2, 1) → (4, 1)`. The **inner dimensions** (both 2) must match, and the **outer dimensions** (4 and 1) give the output shape.


### Read matrix multiplication one row at a time

The first row of `X` is `[1,1]`. Dot it with weights `[2,1]` to obtain `1×2 + 1×1 = 3`. The second row `[2,1]` gives `2×2+1×1=5`, then rows 3 and 4 produce 7 and 9.

**Why is this useful?** One matrix operation makes *four predictions* without writing four separate formulas. A matrix shaped `(number_of_examples, number_of_features)` multiplied by a weight column shaped `(number_of_features, 1)` gives one prediction per example. If the numbers of features do not agree, the multiplication is undefined.


<picture>
  <source type="image/svg+xml" srcset="images/10-matrix-vector.svg">
  <img src="images/10-matrix-vector.png" alt="A 4 × 2 feature matrix times a 2 × 1 weight vector producing all four predictions in a 4 × 1 output grid" loading="lazy">
</picture>

*Figure: A 4 × 2 feature matrix times a 2 × 1 weight vector producing all four predictions in a 4 × 1 output grid.*


In [ ]:
X = torch.tensor([[1.,1.], [2.,1.], [3.,1.], [4.,1.]])
weights = torch.tensor([[2.], [1.]])
predictions = X @ weights
print('X shape:',tuple(X.shape))
print('Weights shape:',tuple(weights.shape))
print('Predictions shape:',tuple(predictions.shape))
print('Predicted scores:',predictions.squeeze(1).tolist())
assert predictions.squeeze(1).tolist() == [3.,5.,7.,9.]


## 4. Matrix–matrix multiplication: what changes?

The same rule works when the second operand is a *matrix* rather than one weight vector. If `A` has shape `(2, 3)` and `B` has shape `(3, 2)`, their product has shape `(2, 2)`.

To find **one output cell**, take a row from `A` and a column from `B` and calculate their dot product. The top-left output, for example, is `1×1 + 2×0 + 3×1 = 4`. The diagram below includes **every** input and output cell. Check another output cell yourself.


### Check every output cell by hand

For `A = [[1,2,3],[4,5,6]]` and `B = [[1,0],[0,1],[1,1]]`, the result is:

| | B column 0 | B column 1 |
|:--|--:|--:|
| A row 0 | `1×1 + 2×0 + 3×1 = 4` | `1×0 + 2×1 + 3×1 = 5` |
| A row 1 | `4×1 + 5×0 + 6×1 = 10` | `4×0 + 5×1 + 6×1 = 11` |

Thus `A @ B = [[4,5],[10,11]]`, with shape `(2,2)`. **Matrix multiplication is not matching-cell multiplication:** it combines a row with a column and sums across the shared inner dimension.


<picture>
  <source type="image/svg+xml" srcset="images/11-matrix-matrix.svg">
  <img src="images/11-matrix-matrix.png" alt="Fully labeled 2 × 3 and 3 × 2 matrices with a complete 2 × 2 result and an annotated cell calculation" loading="lazy">
</picture>

*Figure: Fully labeled 2 × 3 and 3 × 2 matrices with a complete 2 × 2 result and an annotated cell calculation.*


In [ ]:
A = torch.tensor([[1.,2.,3.],[4.,5.,6.]])
B = torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
C_result = A @ B
print(C_result)
print('Result shape:',tuple(C_result.shape))
assert C_result.tolist() == [[4.,5.],[10.,11.]]


### Three symbols to remember

| Python expression | Meaning | Example result |
|:--|:--|:--|
| `a + b` | add matching entries (or broadcast) | `[1,2]+[3,4] → [4,6]` |
| `a * b` | multiply matching entries (or broadcast) | `[1,2]*[3,4] → [3,8]` |
| `A @ B` | row–column dot products | `(2,3)@(3,2) → (2,2)` |

This distinction becomes critical when reading neural-network formulas like `X @ W + b`.

## 5. Reductions: turning many values into fewer values

A **reduction** summarizes a collection. `sum()` adds; `mean()` averages. Along `axis=0`, we summarize **down the rows**, getting one value per column. Along `axis=1`, we summarize **across columns**, getting one value per row.

For the matrix `[[1,2],[3,4]]`, column sums are `[4,6]`; row sums are `[3,7]`.


### Which axis disappears?

Take `[[1,2],[3,4]]`, shape `(2,2)`. `sum(dim=0)` adds **down** each column and returns `[4,6]`. `sum(dim=1)` adds **across** each row and returns `[3,7]`. A *reduction* decreases the number of axes or their sizes by summarizing values.

**ML connection:** averaging a batch of losses turns one loss per example into one batch-level training objective. Choosing the wrong axis can silently produce the wrong result even if the program runs.


In [ ]:
small = torch.tensor([[1., 2.], [3., 4.]])
print('All entries sum:', small.sum().item())
print('Column sums (dim=0):', small.sum(dim=0).tolist())
print('Row sums (dim=1):', small.sum(dim=1).tolist())
print('Mean of all entries:',small.mean().item())


## 6. Norms: measuring vector size

The **L2 norm** measures a vector's length. For `v = [3, 4]`, picture a right triangle with horizontal side 3 and vertical side 4. Pythagoras tells us the diagonal has length 5.

\[\|v\|_2=\sqrt{3^2+4^2}=5.\]

This matters because we often want to measure the size of a parameter vector, error vector, or gradient. You do not need advanced geometry to recognize what a norm tells you.


### Norm versus distance

For `v = [3,4]`, the **L2 norm** is `√(3²+4²)=5`: distance from the origin. To measure the distance *between* two vectors `a` and `b`, first take the difference `a−b`, then its norm `||a−b||₂`.

**Why we care:** gradient clipping and weight regularization can use norms to control the magnitude of parameter vectors or updates; a norm measures size, not whether a prediction is correct.


<picture>
  <source type="image/svg+xml" srcset="images/12-vector-norm.svg">
  <img src="images/12-vector-norm.png" alt="A 3–4–5 right triangle illustrating the length of a vector with entries 3 and 4" loading="lazy">
</picture>

*Figure: A 3–4–5 right triangle illustrating the length of a vector with entries 3 and 4.*


In [ ]:
v = torch.tensor([3.,4.])
print('L2 norm:', torch.linalg.vector_norm(v).item())
print('Squared entries:',(v*v).tolist())
assert torch.linalg.vector_norm(v).item() == 5.


## 7. Small experiment: change one weight

Our simple predictor uses `[weight_for_hours, bias] = [2,1]`. What if the Hours weight becomes **3**? Higher hours would have more effect on the prediction, because they are multiplied by a larger number.

Try predicting the four scores **before** running the cell. (They will no longer match the original synthetic targets; that is useful to notice.)


In [ ]:
new_weights = torch.tensor([[3.],[1.]])
changed_predictions = X @ new_weights
print('Previous:',predictions.squeeze().tolist())
print('New:',changed_predictions.squeeze().tolist())


### Check your understanding

1. Calculate `[3, 2] · [2, 5]` by hand. Is the result a number or a vector?
2. What output shape should you expect from `(5, 3) @ (3, 2)`? Why?
3. When is `A * B` **not** the same as `A @ B`?
4. What is the L2 norm of `[0, 6, 8]`?
5. In the prediction `2×Hours + 1`, which part is the **bias**?

**Takeaway:** Linear algebra lets models combine many numerical features using weighted sums, while tensor shape tells us exactly which calculations are valid.

➡️ **Next:** [2.4 Calculus and automatic differentiation](04-calculus-and-autograd.ipynb) — how does the model know which weight to change?


---



# 2.4 · Calculus and automatic differentiation

**Estimated time:** 45–55 minutes · **Prerequisite:** Lesson 2.3 (a weight, a prediction, and an error).

### What you will understand by the end

- A **derivative** describes how an output changes when an input changes.
- A **gradient** collects partial derivatives for several adjustable parameters.
- The **chain rule** tells us how changes flow through a sequence of calculations.
- A **forward pass** computes a prediction and loss; a **backward pass** computes gradients.
- PyTorch **autograd** computes the derivatives that we would otherwise calculate by hand.

**Our continuous example:** A learner studied `x = 2` hours and the target score is `5`. The toy model predicts `ŷ = w×x + b`, where we start with `w = 1`, `b = 1`.

The example intentionally uses one weight and one training point so you can check **every number**. A real model typically has many parameters and examples.


## 1. Derivatives: how sensitive is an output?

Start with the simple rule `f(x) = x²`.

At `x = 2`, the output is `f(2) = 4`. If we increase x a little, the output rises. A **derivative** measures the rate of change *at a particular input*. Geometrically, it is the slope of the tangent line to the curve.

For this rule, `f′(x) = 2x`. At `x = 2`, the derivative is **4**. That means a tiny increase of `0.01` in x changes f by roughly `4×0.01 = 0.04` near that point.

**The derivative is not the output.** At x=2 the output is 4 and the slope is also 4, but that numerical coincidence is specific to this example.


### What does a derivative answer?

The function answers **"What is the output right now?"** The derivative answers **"If I nudge the input slightly, how quickly does the output change?"** For `f(x)=x²`, at `x=2` the output is 4 and the derivative is 4. Near this point, increasing x by `0.01` raises f by approximately `4×0.01=0.04`.

**Why ML needs this:** a model adjusts weights rather than guessing endlessly. Derivatives describe how changing those weights will influence its error. The slope may be positive, negative, or zero, and is local: it need not stay the same for large changes.


<picture>
  <source type="image/svg+xml" srcset="images/13-derivative-tangent.svg">
  <img src="images/13-derivative-tangent.png" alt="The curve f(x)=x² and its tangent at x=2, highlighting that the derivative is its local slope" loading="lazy">
</picture>

*Figure: The curve f(x)=x² and its tangent at x=2, highlighting that the derivative is its local slope.*


In [ ]:
import torch

# Approximate the derivative from a small change in x.
def f(x):
    return x**2

x = 2.0
for h in [0.1, 0.01, 0.001]:
    approximate_derivative = (f(x+h)-f(x))/h
    print(f'h={h:5g} → approximate slope={approximate_derivative:.3f}')
print('Exact slope from 2*x:', 2*x)


### A gradient is several slopes collected together

When a loss depends on both `w` and `b`, we can ask two questions: how would the loss change if **only w** changed, and how would it change if **only b** changed? Those are partial derivatives. Combining them gives the **gradient** `∇L = [∂L/∂w, ∂L/∂b]`.

We calculate these values *at the current parameter settings*. A gradient says which small changes would initially raise or lower the loss; it is **not** a prediction or a parameter update.


As **h gets smaller**, the measured slope approaches 4. This is the idea behind the limit definition of a derivative. You do not need to derive all the calculus rules here; understanding what they *measure* is the important first step.

## 2. Partial derivatives and gradients

A prediction usually depends on several quantities, such as **weight `w`** and **bias `b`**. A **partial derivative** tells us how the result changes when one of these is changed while the others are held fixed.

For `ŷ = w×x + b`, treating x as a given input:

- `∂ŷ/∂w = x`: how the prediction changes with the weight.
- `∂ŷ/∂b = 1`: how the prediction changes with the bias.

If a loss depends on many parameters, we collect the partial derivatives into a **gradient**: `[∂L/∂w, ∂L/∂b, …]`. Each number tells us which way the loss would change if we adjusted its corresponding parameter.

## 3. First do the forward calculation

We want a prediction close to the target `5`. With `x=2`, `w=1` and `b=1`:

1. **Prediction:** `ŷ = 1×2 + 1 = 3`.
2. **Difference:** `ŷ - target = 3 - 5 = -2`.
3. **Squared-error loss:** `L = (-2)² = 4`.

A **loss function** converts the model's prediction error into a number that training tries to reduce. This simple squared error is not the only possible loss, but it makes the arithmetic easy to inspect.


<picture>
  <source type="image/svg+xml" srcset="images/14-forward-graph.svg">
  <img src="images/14-forward-graph.png" alt="Four connected stages of a forward pass: input, prediction, difference, and squared error" loading="lazy">
</picture>

*Figure: Four connected stages of a forward pass: input, prediction, difference, and squared error.*


In [ ]:
hours = torch.tensor(2.)
target = torch.tensor(5.)
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(1., requires_grad=True)

prediction = w*hours + b
loss = (prediction - target)**2

print('Prediction:',prediction.item())
print('Squared error:',loss.item())
print('Gradient BEFORE backward:',w.grad)
assert prediction.item()==3. and loss.item()==4.


Notice that `w.grad` is `None` here. The **forward pass** calculates the **value** of the prediction and loss, but it has not yet calculated the derivatives. PyTorch records operations involving `requires_grad=True` tensors so it can differentiate them later.

## 4. The chain rule explains the backward calculation

Our loss depends on `w` **through** the prediction:

\[w\;\longrightarrow\;\hat y = 2w+1\;\longrightarrow\;L=(\hat y-5)^2.\]

The **chain rule** says: multiply the slope of the loss with respect to the prediction by the slope of the prediction with respect to the weight:

\[\frac{dL}{dw} = \frac{dL}{d\hat y}\times\frac{d\hat y}{dw}.\]

At the starting point:

- `dL/dŷ = 2(ŷ - 5) = 2(3 - 5) = -4`.
- `dŷ/dw = x = 2`.
- So **`dL/dw = -4 × 2 = -8`**.

For bias, the slope is `dŷ/db = 1`, so `dL/db = -4 × 1 = -4`.

A *negative* derivative with respect to w means that **a small increase in w** would initially **reduce** the loss.


### Follow the complete chain backward

| Stage | Forward value | Local derivative needed for `dL/dw` |
|:--|:--|:--|
| Weight/input | `w=1`, `x=2`, `b=1` | `dŷ/dw=x=2` |
| Prediction | `ŷ=wx+b=3` | next: `dL/dŷ=2(ŷ−5)` |
| Difference | `ŷ−5=−2` | `2×(−2)=−4` |
| Squared loss | `L=(−2)²=4` | multiply `−4×2=−8` |

The loss gradient with respect to `w` is `−8`: increasing `w` slightly initially decreases loss. **The chain rule multiplies local derivatives along a path**, rather than multiplying outputs from successive forward passes. For the bias, `dŷ/db=1`, so `dL/db=−4`.


<picture>
  <source type="image/svg+xml" srcset="images/15-backward-chain-rule.svg">
  <img src="images/15-backward-chain-rule.png" alt="The chain rule calculation leading to gradient −8 and a weight update from 1 to 1.8" loading="lazy">
</picture>

*Figure: The chain rule calculation leading to gradient −8 and a weight update from 1 to 1.8.*


In [ ]:
loss.backward()  # run automatic differentiation
print('dL/dw:',w.grad.item())
print('dL/db:',b.grad.item())
assert w.grad.item()==-8.0
assert b.grad.item()==-4.0


### What did `backward()` actually do?

It **computed derivatives** and stored them in `.grad`. It did **not** update `w` or `b`. In neural-network training, computing gradients (backpropagation) and **using** those gradients to change parameters (optimization) are separate steps.

**One controlled update:** To see the effect of a gradient without changing two parameters simultaneously, we will update **only the weight** with learning rate `0.1`, holding b fixed at 1.

\[w_{new} = w_{old} - 0.1\frac{dL}{dw} = 1 - 0.1(-8) = 1.8.\]

A **learning rate** controls how large a step we take. The updated prediction is `2×1.8 + 1 = 4.6`, and squared error is `(4.6-5)² = 0.16`—smaller than 4.


### Distinguish three separate actions

1. **Forward propagation:** evaluate `ŷ = wx+b` and the loss, producing ordinary numerical values.
2. **Backward propagation (backpropagation):** traverse the recorded computation graph using the chain rule and accumulate derivatives in `.grad`.
3. **Optimizer step:** use those derivatives to **change** parameter values. A common update is `new_weight = old_weight − learning_rate × gradient`.

Autograd is the software system that computes derivatives; it does not decide the learning rate or update weights on its own. Forward-mode automatic differentiation is another derivative-computation method and is **not** just running the model's forward pass.


<picture>
  <source type="image/svg+xml" srcset="images/16-gradient-step.svg">
  <img src="images/16-gradient-step.png" alt="The loss curve showing how updating the weight from 1.0 to 1.8 reduces squared error from 4 to 0.16" loading="lazy">
</picture>

*Figure: The loss curve showing how updating the weight from 1.0 to 1.8 reduces squared error from 4 to 0.16.*


In [ ]:
learning_rate = 0.1
with torch.no_grad():  # this is an update, not a step we want to differentiate
    w -= learning_rate*w.grad

new_prediction = w*hours + b
new_loss = (new_prediction - target)**2
print('New weight:',round(w.item(),3))
print('New prediction:',round(new_prediction.item(),3))
print('New loss:',round(new_loss.item(),4))
assert abs(new_loss.item() - 0.16) < 1e-5


## 5. Why gradients need to be cleared

By default, PyTorch **accumulates** gradients into `.grad`: calling backward a second time can **add** derivatives rather than replacing the old ones. Training loops therefore typically clear gradients before the next backward pass (for example `optimizer.zero_grad()` or `w.grad.zero_()`).

The small demonstration below creates a **fresh weight** to avoid confusing its derivative with any previous calculation.


### Why can `.grad` surprise you?

Suppose one backward pass yields a gradient of `−8`. If we run another backward pass that contributes `−8`, PyTorch may leave `−16` in the same `.grad` buffer, because gradients **accumulate** by default. That is useful in some advanced training setups but misleading if you thought `.grad` would always be overwritten.

A normal training loop therefore clears gradients, calculates a new loss, calls `backward()`, and then updates parameters. **Clear → forward → backward → update** is a helpful beginner mental model.


In [ ]:
fresh_w = torch.tensor(1.,requires_grad=True)
first_loss = (2*fresh_w + 1 - 5)**2
first_loss.backward()
print('First gradient:',fresh_w.grad.item())

fresh_w.grad.zero_()
second_loss = (2*fresh_w + 1 - 5)**2
second_loss.backward()
print('After clearing, second gradient:',fresh_w.grad.item())


## 6. Commonly confused terms

| Term | What it means | What it **does not** mean |
|:--|:--|:--|
| **Forward propagation** | The model calculates its prediction (and loss) from current inputs/parameters. | It is not a parameter update. |
| **Automatic differentiation (autograd)** | Software calculates derivatives of code operations. | It is not the optimizer itself. |
| **Backpropagation** | A reverse pass applies the chain rule to calculate gradients of a loss with respect to parameters. | It does not automatically update weights. |
| **Gradient descent** | A procedure uses a gradient to change parameters, usually toward lower loss. | It is not the derivative calculation itself. |

**Forward-mode automatic differentiation** is another method for computing derivatives. It is **not the same thing** as “forward propagation.” We do not need forward-mode details for this beginner lesson.

## 7. Your turn: predict the derivative

If `L = (2w + 1 - 5)²`, what is `dL/dw` at `w = 2`? Check by hand **before** running the following cell. What does the derivative tell you about where the minimum might be?


In [ ]:
test_w = torch.tensor(2.,requires_grad=True)
test_loss = (2*test_w + 1 - 5)**2
test_loss.backward()
print('Loss at w=2:',test_loss.item())
print('Gradient at w=2:',test_w.grad.item())


### Check your understanding

1. Is a derivative the output of the function or the output's *rate of change*?
2. In the toy example, why is `w.grad` initially `None`?
3. What are the two multiplications involved in the **chain rule** that give `dL/dw = -8`?
4. Why do we write `with torch.no_grad():` around the manual update?
5. Does `.backward()` change parameters directly? If not, what changes them?

**Takeaway:** Training a model involves calculating a loss, finding how sensitive it is to the parameters (gradients), and applying updates that aim to reduce that loss. This is the beginning of **backpropagation**, the mechanism used to train neural networks.

➡️ **Next:** [2.5 Probability and statistics](05-probability-and-statistics.ipynb) — why individual outcomes and datasets are uncertain.


---



# 2.5 · Probability and statistics: reasoning about uncertainty

**Estimated time:** 35–45 minutes · **Prerequisite:** counting, fractions and running simple Python code.

### By the end, you should be able to

- Explain a **random experiment**, an **outcome**, and an **event**.
- Interpret **probability** as a number between 0 and 1.
- Compare **theoretical probability** with **observed relative frequency**.
- See why estimates become more stable as sample sizes grow (without expecting them to be perfectly equal).
- Compute a simple **conditional probability** and basic summaries such as the mean.

We first use coin tosses because we know the ideal mathematical probabilities. Then we return to our **four fictional learners** and show why tiny datasets require caution.


## 1. Probability is about uncertain outcomes

Toss a fair coin. There are two possible outcomes: **Heads (H)** and **Tails (T)**. A fair coin assigns each outcome probability `1/2 = 0.5 = 50%`.

An **event** is a set of outcomes we care about. If we toss a coin *twice*, the four equally likely outcomes are HH, HT, TH and TT. The event “exactly one head” includes **HT and TH**, so:

\[P(\text{exactly one head}) = 1/4 + 1/4 = 1/2.\]

These calculations assume **fair and independent** tosses. If the coin is biased, the probabilities change.


### What, why, how, and where?

**What:** Probability quantifies how likely an event is under an assumed process or model. A probability of 0 means impossible under that model; 1 means certain. **Why:** predictions and real-world observations can be uncertain even when we have good data.

**How:** define the experiment and event first. With two independent fair coin tosses, there are four equally likely outcomes: HH, HT, TH, TT. The event "exactly one head" is `{HT, TH}`, so its probability is `2/4 = 0.5`. **ML connection:** probability helps us interpret prediction uncertainty and model evaluation, but an assumption like "fair independent tosses" must always be stated.


<picture>
  <source type="image/svg+xml" srcset="images/17-coin-outcomes.svg">
  <img src="images/17-coin-outcomes.png" alt="A two-toss probability tree showing all four outcomes and probability one-quarter for each" loading="lazy">
</picture>

*Figure: A two-toss probability tree showing all four outcomes and probability one-quarter for each.*


**Interpretation:** A probability of `0.5` does not promise that the first 10 tosses will produce *exactly* five heads. It describes the long-run behavior of the ideal random process, not a guaranteed count in a small sample.

## 2. Observed frequency is an estimate

We can *simulate* tossing a fair coin. Each trial produces either `0` (tails) or `1` (heads). The **relative frequency** of heads is:

\[\frac{\text{number of observed heads}}{\text{total tosses}}.\]

This is an **estimate** from the results we observed. It need not equal the theoretical probability exactly, even with many tosses.

We use a fixed random seed so the outputs are reproducible on different learners' machines.


### Theoretical probability is not an observed count

For a fair coin, `P(heads)=0.5` describes the generating process. An observed fraction—for example, 7 heads in 10 tosses, or `7/10=0.7`—describes one **sample**. The two can differ without any contradiction.

**Why more data helps:** the running fraction tends to settle *near* 0.5 as we collect many independent fair tosses; it need not hit 0.5 exactly. A larger sample reduces the influence of any one unusual result, but the correctness of an estimate still depends on assumptions and sampling.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
tosses = rng.integers(0,2,size=1000)  # 0 = tails, 1 = heads

for n in [10,100,1000]:
    heads = int(tosses[:n].sum())
    print(f'First {n:4} tosses: {heads:4} heads; fraction = {heads/n:.3f}')


Now look at the **complete running estimate**, not just the three snapshots. The orange line shows the theoretical probability 0.5, while the teal line records the fraction observed as we collect more tosses.


<picture>
  <source type="image/svg+xml" srcset="images/18-sample-size.svg">
  <img src="images/18-sample-size.png" alt="A reproducible plot of the running observed fraction of heads across 1000 simulated coin tosses, with a 0.5 reference line" loading="lazy">
</picture>

*Figure: A reproducible plot of the running observed fraction of heads across 1000 simulated coin tosses, with a 0.5 reference line.*


In [ ]:
running_fraction = np.cumsum(tosses) / np.arange(1,len(tosses)+1)
fig,ax=plt.subplots(figsize=(8,3.7))
ax.plot(np.arange(1,len(tosses)+1),running_fraction,label='Observed fraction')
ax.axhline(.5,linestyle='--',label='Fair coin probability (0.5)')
ax.set(xlabel='Toss count',ylabel='Fraction of heads',title='Try changing the random seed or number of tosses')
ax.legend();plt.show()


### What larger samples really mean

Larger samples **usually** make an empirical proportion more stable, because a few unusual outcomes have less influence. This is an example of the **law of large numbers**. However:

- The proportions do **not** become exactly equal to 0.5 after a fixed number of tosses.
- The actual **counts** of heads and tails do **not** need to be equal.
- The process must be appropriate for the assumptions: a biased coin will stabilize around its own probability, not 0.5.

This matters in ML because a model's evaluation score on a **small test set** can be noisy.


## 3. A conditional probability asks a more specific question

Return to the toy class:

| Learner | Hours | Score | Passed (score ≥ 6)? |
|:--|--:|--:|:--|
| Alice | 1 | 3 | No |
| Ben | 2 | 5 | No |
| Chika | 3 | 7 | Yes |
| Dara | 4 | 9 | Yes |

Across all four learners, the **observed fraction** who passed is `2/4 = 0.5`. But perhaps we ask: **Among learners who studied at least 3 hours, what fraction passed?**

Only Chika and Dara are in that group, and both passed. So **`P(pass | Hours ≥ 3)`**, as *estimated from this toy sample*, is `2/2 = 1.0`.

The vertical bar `|` reads **“given that.”** It restricts which rows we count in the denominator.


### Conditional probability changes which rows we count

A conditional probability restricts attention to cases where some condition is true. Its core rule is `P(A | B) = P(A and B) / P(B)`, provided `P(B) > 0`. In a dataset, you can think of this as:

> Among **only the rows where B is true**, what fraction also satisfy A?

In our four-learner sample, `B` = "studied at least 3 hours" includes Chika and Dara (two rows). `A` = "passed" is true for both, so `P(pass | Hours≥3)` is estimated as `2/2=1` **in this invented sample**. It does not establish that studying caused passing. If no rows meet B, the observed fraction is undefined rather than zero.


<picture>
  <source type="image/svg+xml" srcset="images/19-conditional-probability.svg">
  <img src="images/19-conditional-probability.png" alt="A fully labeled count table for pass versus not pass grouped by study hours, including row/column totals" loading="lazy">
</picture>

*Figure: A fully labeled count table for pass versus not pass grouped by study hours, including row/column totals.*


In [ ]:
import pandas as pd

sample = pd.DataFrame({
    'Learner':['Alice','Ben','Chika','Dara'],
    'Hours':[1,2,3,4],
    'Score':[3,5,7,9],
})
sample['Passed'] = sample['Score']>=6
sample['Hours_3_or_more'] = sample['Hours']>=3

print('Overall pass fraction:',sample['Passed'].mean())
print('Pass fraction given Hours>=3:',
      sample.loc[sample['Hours_3_or_more'],'Passed'].mean())
print('\nCounts:\n',pd.crosstab(sample['Passed'],sample['Hours_3_or_more'],margins=True))


**Critical caution:** These are only **four invented observations**. The conditional estimate `1.0` is not a law about students, a reliable estimate for the wider world, or proof that studying three hours *causes* a pass. Statistical relationships can reflect other differences between groups. The difference between **observation** and **causal explanation** will matter in later ML and AI safety evaluation.

## 4. Mean and spread: two basic statistical summaries

A **mean** (average) tells us the center of a group of numbers. For our scores `[3,5,7,9]`, the mean is `(3+5+7+9)/4 = 6`.

But the mean hides **variation**: learners scored differently. Standard deviation gives one way to summarize **spread**. For this beginner course, you only need to understand that a larger standard deviation usually means the values vary more around the mean. We'll use PyTorch to calculate it.

Be careful when interpreting a statistic: a single mean or pass rate is not a complete description of a dataset.


### Same mean, different variability

Two classes might have scores `[5,5,5]` and `[3,5,7]`. Both have mean 5, but the first has **no spread** while the second varies. Standard deviation summarizes typical variation around the mean (in the original units).

**Why this matters:** reporting only average model accuracy can conceal unreliable behavior on some examples. Also check variability, subgroup differences, and sample size; a single statistic never tells the whole story.


In [ ]:
scores = torch.tensor([3.,5.,7.,9.])
print('Average score:',scores.mean().item())
print('Minimum and maximum:',scores.min().item(),scores.max().item())
print('Population standard deviation:',round(scores.std(unbiased=False).item(),3))

second_group=torch.tensor([6.,6.,6.,6.])
print('Another group with same mean:',second_group.mean().item())
print('But different spread:',second_group.std(unbiased=False).item())


## 5. Why probability matters for machine learning and AI safety

Models make predictions on imperfect observations, under uncertainty. When evaluating models, we often report fractions such as **accuracy** or **failure rate**. These numbers depend on which examples were tested, how many examples were included, and what we counted as success.

Even if two models have the same accuracy on a tiny set, that alone does not show they have the same reliability across different situations. We need to be able to discuss **sampling, uncertainty, and distribution of outcomes** rather than treating one result as absolute certainty.

The purpose of this lesson is *not* to master all of statistics. It is to develop the habits of asking: “What happened?”, “How often?”, “Among whom?”, and “How confident should we be?”

## 6. Small experiment: compare different random seeds

Re-run the following cell using seed `0`, `1`, and `2`. Do the first 10 flips always have the same fraction of heads? Do the first 1000 fractions become identical?


In [ ]:
for seed in [0,1,2]:
    flips = np.random.default_rng(seed).integers(0,2,size=1000)
    print(f'seed {seed}: first 10 = {flips[:10].mean():.2f}, all 1000 = {flips.mean():.3f}')


### Check your understanding

1. What is the difference between theoretical probability and the **fraction observed** in a sample?
2. For two fair independent tosses, what is the probability of **at least one head**?
3. Why should you not expect **exactly** 500 heads in every set of 1,000 fair-coin tosses?
4. What does `P(pass | Hours ≥ 3)` mean—and which learners are in its denominator?
5. Two groups have the same mean score. Does that guarantee that they have the same spread?

**Takeaway:** Probability gives a way to reason about uncertain events; statistics helps us describe the outcomes that actually occurred. Both are essential for interpreting a model's behavior responsibly.

➡️ **Next:** [Chapter 2 checkpoint](quizzes/chapter-02-checkpoint.ipynb). Chapter 3 (Machine Learning Basics) is the next planned chapter.


---



# Chapter 2 quick reference · Finding help in Python and PyTorch

You do **not** need to memorize every PyTorch function. Much more important is knowing how to inspect a function and verify what it expects.

- `help(torch.ones)` shows documentation for making a tensor of ones.
- `torch.ones(2, 3)` creates a tensor with **2 rows and 3 columns**.
- `dir(torch)` lists available names, which is useful for discovery but not something to memorize.
- In Jupyter, `torch.sum?` also shows help in an interactive notebook (the standard Python `help(...)` works everywhere).

These examples are adapted in scope from the uploaded D2L **Documentation** notebook.


In [ ]:
import torch
print('A 2 × 3 tensor of ones:\n',torch.ones(2,3))
print('Shape:',tuple(torch.ones(2,3).shape))
print('Does torch have a mean function?',hasattr(torch,'mean'))


## When stuck, check three things

1. **Shape:** What dimensions does the function expect and return?
2. **Type:** Do inputs need to be floats, integers, tensors, or Python lists?
3. **Meaning:** Does the operation act on each element, each row/column, or the entire tensor?

Practise by reading the documentation string rather than copying unfamiliar code blindly.


In [ ]:
# Prints the first lines of help without overwhelming the notebook.
first_lines = (torch.mean.__doc__ or '').strip().splitlines()[:13]
print('\n'.join(first_lines))


## A practical habit for learning new operations

Whenever you find an unfamiliar function, ask **four questions**: (1) What does it compute? (2) Why would we use it? (3) Which input shapes/types does it accept, and what shape does it return? (4) Where will it be useful in ML?

For example, documentation for `torch.matmul` helps distinguish *matrix multiplication* from `torch.mul`, and `torch.broadcast_shapes` clarifies shape compatibility. Reading expected inputs and outputs is often more useful than memorizing the function name.


## Key terms from Chapter 2

| Term | Plain meaning |
|:--|:--|
| **Tensor** | A regular arrangement of values, with a defined shape. |
| **Feature** | A model input such as study hours. |
| **Target** | The desired prediction (for example a quiz score). |
| **Dot product** | Multiply corresponding vector entries, then add. |
| **Gradient** | Derivatives showing how a loss changes with model parameters. |
| **Autograd** | Automatic calculation of those derivatives. |
| **Probability** | A mathematical way to describe how likely an uncertain event is. |

**Next:** Return to the [Chapter 2 overview](README.md), or complete the [checkpoint](quizzes/chapter-02-checkpoint.ipynb).
